In [10]:
from ortools.sat.python import cp_model

def planificar_mudanza():
  # Crear el modelo de CP
  modelo = cp_model.CpModel()

  # Datos del problema
  muebles = ['Piano', 'Cama', 'Silla', 'Mesa']
  personas = [3, 3, 1, 2]
  duraciones = [30, 15, 10, 15]
  num_muebles = len(muebles)
  horizonte = 60  # Tiempo máximo en minutos

  # Variables de inicio para cada mueble
  inicio = [modelo.NewIntVar(0, horizonte, f'inicio_{muebles[i]}') for i in range(num_muebles)]
  
  # Variables de fin para cada mueble
  fin = [modelo.NewIntVar(0, horizonte, f'fin_{muebles[i]}') for i in range(num_muebles)]
  
  # Crear variables de intervalo (IntervalVar)
  intervalos = [
    modelo.NewIntervalVar(
      start=inicio[i],
      size=duraciones[i],
      end=fin[i],
      name=f'intervalo_{muebles[i]}'
    )
    for i in range(num_muebles)
  ]

  # Restricción: fin = inicio + duración
  for i in range(num_muebles):
    modelo.Add(fin[i] == inicio[i] + duraciones[i])

  # Restricción Cumulative: no exceder 4 personas en ningún momento
  modelo.AddCumulative(
    intervals=intervalos,
    demands=personas,
    capacity=4
  )

  # Objetivo: minimizar el tiempo máximo de finalización (makespan)
  makespan = modelo.NewIntVar(0, horizonte, 'makespan')
  modelo.AddMaxEquality(makespan, fin)
  modelo.Minimize(makespan)
  
  # Resolver el modelo
  solver = cp_model.CpSolver()
  status = solver.Solve(modelo)
  # Mostrar resultados
  if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("Horario factible encontrado:")
    print("Mueble\tInicio\tFin\tPersonas\tDuración")
    for i in range(num_muebles):
      print(f"{muebles[i]}\t{solver.Value(inicio[i])}\t{solver.Value(fin[i])}\t{personas[i]}\t\t{duraciones[i]}")
    print(f"\nTiempo total de finalización (makespan): {solver.Value(makespan)} minutos")
  else:
    print("No se encontró una solución factible.")

# Ejecutar la función
planificar_mudanza()

Horario factible encontrado:
Mueble	Inicio	Fin	Personas	Duración
Piano	0	30	3		30
Cama	45	60	3		15
Silla	0	10	1		10
Mesa	30	45	2		15

Tiempo total de finalización (makespan): 60 minutos
